# Import Libraries

In [ ]:
import csv
import pylidc as pl
import pandas as pd
from tqdm import tqdm

In [ ]:
SCAN_METADATA_PATH = "../datasets/metadata/scan_metadata.csv"

# Query All Images

`scans`
- dtype: <class 'sqlalchemy.orm.query.Query'>
- is a query plan, isn't executed yet
- .yield_per() is for batching

In [ ]:
scans = pl.query(pl.Scan).yield_per(10)
num_data = pl.query(pl.Scan).count()
session = scans.session

print(f"total data: {num_data}")

In [ ]:
fieldnames = [
    "patient_id",
    "study_instance_uid",
    "series_instance_uid",
    "slice_thickness",
    "pixel_spacing_x",
    "pixel_spacing_y",
    "num_slices",
    "num_annotations",
    "num_nodules",
    "num_annotations_per_nodule",
    "annotation_ids_per_nodule",
    "contrast_used"
]

with open(SCAN_METADATA_PATH, mode="w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for scan in tqdm(scans, total=num_data, desc="Extracting scan metadata"):
        try:
            # --- pixel spacing ---
            if isinstance(scan.pixel_spacing, (list, tuple)) and len(scan.pixel_spacing) >= 2:
                px, py = scan.pixel_spacing[:2]
            elif scan.pixel_spacing is not None:
                px = py = scan.pixel_spacing
            else:
                px = py = None

            # --- slice ---
            num_slices = len(scan.slice_zvals) if scan.slice_zvals is not None else None

            # --- annotations ---
            num_annotations = len(scan.annotations)

            # --- contrast ---
            contrast_used = getattr(scan, "contrast_used", None)

            # --- clustering ---
            try:
                nods = scan.cluster_annotations()
                num_nodules = len(nods)
                num_annotations_per_nodule = [len(nod) for nod in nods]
                annotation_ids_per_nodule = [[ann.id for ann in nod] for nod in nods]
            except Exception:
                num_nodules = None
                num_annotations_per_nodule = None
                annotation_ids_per_nodule = None
                session.rollback()

            record = {
                "patient_id": scan.patient_id,
                "study_instance_uid": scan.study_instance_uid,
                "series_instance_uid": scan.series_instance_uid,
                "slice_thickness": scan.slice_thickness,
                "pixel_spacing_x": px,
                "pixel_spacing_y": py,
                "num_slices": num_slices,
                "num_annotations": num_annotations,
                "num_nodules": num_nodules,
                "num_annotations_per_nodule": str(num_annotations_per_nodule),
                "annotation_ids_per_nodule": annotation_ids_per_nodule,
                "contrast_used": contrast_used
            }

            writer.writerow(record)

        except Exception as e:
            print(f"Error at scan {scan.patient_id}: {e}")
            session.rollback()
            continue

print(f"Saved to {SCAN_METADATA_PATH}")